# Seminar 10: Object Detection and YOLO Output Analysis

**Student Version**

This seminar introduces practical object detection before the YOLO homework.

Goals for today:
- Understand detection outputs: boxes, classes, confidence scores
- See why box format mistakes break visualizations
- Implement IoU from scratch
- Implement simple NMS on toy boxes
- Run YOLO inference and plot boxes manually
- Analyze confidence and NMS thresholds in YOLO outputs


## 0. Theory: From Classification to Detection

Classification returns one label for the whole image.

Object detection returns a list of objects. Each detection usually contains:

```text
class_id, confidence, bounding_box
```

The list length can change from image to image.

A bounding box is often stored as:

```text
xyxy = [x_min, y_min, x_max, y_max]
```

or:

```text
xywh = [x_center, y_center, width, height]
```

Always check the format before drawing or computing overlap.

### IoU

Intersection over Union measures overlap between two boxes:

```text
IoU = area(intersection) / area(union)
```

Interpretation:
- `0`: no overlap,
- `1`: perfect overlap.

IoU is used for evaluation and for removing duplicate detections.

### NMS

Detectors often predict several boxes around the same object.

Non-Maximum Suppression keeps the highest-confidence box and removes weaker boxes that overlap too much with it.

Simplified NMS:

```text
1. sort boxes by confidence
2. keep the highest-confidence box
3. remove lower-confidence boxes with IoU above threshold
4. repeat
```

YOLO uses this type of post-processing to turn many candidate predictions into final detections.


## 1. Setup

Optional install for Colab if needed:

```python
%pip install ultralytics
```


In [ ]:
from io import BytesIO

import pandas as pd
import requests
from PIL import Image, ImageDraw

import torch
import matplotlib.pyplot as plt

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None
    print('ultralytics is not installed. In Colab, run: %pip install ultralytics')


### Config

In [ ]:
CFG = {
    'model_name': 'yolov8n.pt',
    'conf': 0.25,
    'iou': 0.7,
    'top_k_show': 8,
}
CFG


### Helpers

In [ ]:
DEFAULT_IMAGE_URLS = [
    # COCO validation images with several objects. These make confidence/NMS effects easier to see.
    'http://images.cocodataset.org/val2017/000000000139.jpg',
    'http://images.cocodataset.org/val2017/000000000785.jpg',
]


def print_shape(name, x):
    if x is None:
        print(f'{name}: not filled yet')
        return
    print(f'{name}: shape={tuple(x.shape)} dtype={x.dtype}')


def load_pil_from_url(url):
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert('RGB')
    return image


def show_images(images, titles=None, figsize=(10, 4)):
    plt.figure(figsize=figsize)
    for i, image in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(image)
        plt.axis('off')
        if titles is not None:
            plt.title(titles[i])
    plt.tight_layout()
    plt.show()


def draw_boxes_on_image(image, boxes, labels=None, scores=None, color='red', width=3):
    image = image.copy()
    draw = ImageDraw.Draw(image)

    for i, box in enumerate(boxes):
        if box is None:
            continue

        x1 = float(box[0])
        y1 = float(box[1])
        x2 = float(box[2])
        y2 = float(box[3])

        left = min(x1, x2)
        top = min(y1, y2)
        right = max(x1, x2)
        bottom = max(y1, y2)
        draw.rectangle([left, top, right, bottom], outline=color, width=width)

        text = ''
        if labels is not None:
            text = str(labels[i])
        if scores is not None:
            score_text = f'{float(scores[i]):.2f}'
            if text == '':
                text = score_text
            else:
                text = text + ' ' + score_text

        if text != '':
            text_y = max(0, top - 12)
            draw.text((left, text_y), text, fill=color)

    return image


def show_yolo_result(result, title=None):
    rendered = result.plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(rendered)
    plt.axis('off')
    if title is not None:
        plt.title(title)
    plt.show()


def summarize_yolo_result(result, max_rows=12):
    boxes = result.boxes
    columns = ['class_id', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2']

    if boxes is None or len(boxes) == 0:
        print('No detections')
        return pd.DataFrame(columns=columns)

    xyxy = boxes.xyxy.detach().cpu()
    class_ids = boxes.cls.detach().cpu().long()
    confidences = boxes.conf.detach().cpu()

    rows = []
    for i in range(len(class_ids)):
        class_id = int(class_ids[i])
        box = xyxy[i]
        rows.append({
            'class_id': class_id,
            'class_name': result.names[class_id],
            'confidence': float(confidences[i]),
            'x1': float(box[0]),
            'y1': float(box[1]),
            'x2': float(box[2]),
            'y2': float(box[3]),
        })

    df = pd.DataFrame(rows, columns=columns)
    print(df.head(max_rows).to_string(index=False))
    return df


def class_labels_from_ids(class_ids, class_names):
    labels = []
    for class_id in class_ids:
        label = class_names[int(class_id)]
        labels.append(label)
    return labels


## 2. Exercise 1: Box Format Visual Debugging

Box format mistakes are easier to understand when we draw them.

Task:
- implement conversion from `xywh` to `xyxy`,
- draw a correct box on a blank image,
- draw the same numbers wrongly interpreted as `xyxy`,
- compare how the two boxes look.

Function contract:
- `xywh_to_xyxy(box)` receives `[x_center, y_center, width, height]`.
- It returns `[x_min, y_min, x_max, y_max]` as a tensor of shape `[4]`.

Useful contract:
- `draw_boxes_on_image(image, boxes, labels=None, scores=None)` expects boxes in `xyxy` format.


In [ ]:
# Exercise 1

def xywh_to_xyxy(box):
    return None

blank_image = Image.new('RGB', (220, 160), color='white')
box_xywh = torch.tensor([60.0, 50.0, 120.0, 90.0])
correct_xyxy = xywh_to_xyxy(box_xywh)
wrong_xyxy = box_xywh

correct_image = draw_boxes_on_image(blank_image, [correct_xyxy], labels=['correct'], color='green')
wrong_image = draw_boxes_on_image(blank_image, [wrong_xyxy], labels=['wrong'], color='red')

# show_images([correct_image, wrong_image], titles=['correct xywh -> xyxy', 'wrong: xywh treated as xyxy'])
print('box_xywh:', box_xywh)
print('correct_xyxy:', correct_xyxy)
print('wrong_xyxy:', wrong_xyxy)


### Checks (Exercise 1)

In [ ]:
assert correct_xyxy.shape == (4,)
assert torch.allclose(correct_xyxy, torch.tensor([0.0, 5.0, 120.0, 95.0]))
assert isinstance(correct_image, Image.Image)
assert isinstance(wrong_image, Image.Image)
print('Exercise 1 passed.')


## 3. Exercise 2: IoU From Scratch

IoU measures how much two boxes overlap.

Formula:

```text
IoU(A, B) = area(A intersection B) / area(A union B)
```

where:

```text
area(A union B) = area(A) + area(B) - area(A intersection B)
```

Task:
- implement IoU for two boxes in `xyxy` format,
- test cases for partial overlap, no overlap, and perfect overlap.

Useful contracts:
- intersection width is `max(0, min(x2 values) - max(x1 values))`,
- intersection height is similar,
- union is `area_a + area_b - intersection_area`.


In [ ]:
# Exercise 2

def box_area_xyxy(box):
    return None


def box_iou_xyxy(box_a, box_b):
    return None

box_a = torch.tensor([0.0, 0.0, 100.0, 100.0])
box_b = torch.tensor([50.0, 50.0, 150.0, 150.0])
box_c = torch.tensor([200.0, 200.0, 300.0, 300.0])

iou_ab = box_iou_xyxy(box_a, box_b)
iou_ac = box_iou_xyxy(box_a, box_c)
iou_aa = box_iou_xyxy(box_a, box_a)

print('IoU(a, b):', iou_ab)
print('IoU(a, c):', iou_ac)
print('IoU(a, a):', iou_aa)


### Checks (Exercise 2)

In [ ]:
assert torch.isclose(iou_ab, torch.tensor(2500.0 / 17500.0))
assert torch.isclose(iou_ac, torch.tensor(0.0))
assert torch.isclose(iou_aa, torch.tensor(1.0))
print('Exercise 2 passed.')


## 4. Exercise 3: Manual NMS on Toy Boxes

NMS removes duplicate detections.

Task:
- sort boxes by score,
- keep the strongest box,
- remove lower-scoring boxes with IoU above the threshold,
- repeat until no boxes remain.

Function contract:
- `manual_nms(boxes, scores, iou_threshold)` receives boxes `[N, 4]` in `xyxy` format and scores `[N]`.
- It returns indices of kept boxes.

Useful hint:
- `torch.argsort(scores, descending=True)` returns indices ordered from highest score to lowest score.


In [ ]:
# Exercise 3

toy_boxes = torch.tensor([
    [20.0, 20.0, 120.0, 120.0],
    [25.0, 25.0, 118.0, 118.0],
    [180.0, 30.0, 280.0, 130.0],
    [185.0, 35.0, 275.0, 125.0],
])
toy_scores = torch.tensor([0.95, 0.80, 0.90, 0.60])

def manual_nms(boxes, scores, iou_threshold):
    keep = []
    return torch.tensor(keep, dtype=torch.long)

keep_low_iou = manual_nms(toy_boxes, toy_scores, iou_threshold=0.3)
keep_high_iou = manual_nms(toy_boxes, toy_scores, iou_threshold=0.9)

print('keep with IoU threshold 0.3:', keep_low_iou)
print('keep with IoU threshold 0.9:', keep_high_iou)


### Checks (Exercise 3)

In [ ]:
assert keep_low_iou.dtype == torch.long
assert keep_high_iou.dtype == torch.long
assert 0 in keep_low_iou.tolist()
assert 2 in keep_low_iou.tolist()
assert len(keep_low_iou) <= len(keep_high_iou)
print('Exercise 3 passed.')


## 5. Provided Demo: Load Images and YOLO

The model and default images are provided so the seminar can focus on interpreting detector outputs.

If `ultralytics` is missing in Colab, run this in a separate cell:

```python
%pip install ultralytics
```


In [ ]:
images = []
for url in DEFAULT_IMAGE_URLS:
    image = load_pil_from_url(url)
    images.append(image)

show_images(images, titles=['COCO image 1', 'COCO image 2'])

if YOLO is None:
    model = None
    print('Install ultralytics before running YOLO inference.')
else:
    model = YOLO(CFG['model_name'])
    print('Loaded model:', CFG['model_name'])


## 6. Exercise 4: Inspect YOLO Results and Plot Boxes Manually

YOLO returns a result object. The important fields are:
- `result.boxes.xyxy`: box coordinates,
- `result.boxes.cls`: predicted class ids,
- `result.boxes.conf`: confidence scores,
- `result.names`: mapping from class id to class name.

Task:
- run YOLO on one image,
- extract boxes/classes/confidences,
- convert class ids to readable labels,
- implement a small manual plotting helper using `draw_boxes_on_image`,
- summarize the result in a table.

Function contract:
- `plot_yolo_boxes(image, boxes_xyxy, class_ids, confidences, class_names, title)` displays boxes over the original image.
- `boxes_xyxy` must have shape `[N, 4]`.
- The text next to each box should include the class name and confidence score.


In [ ]:
# Exercise 4

result = None
boxes_xyxy = None
class_ids = None
confidences = None
class_labels = None
result_df = None


def plot_yolo_boxes(image, boxes_xyxy, class_ids, confidences, class_names, title=None):
    labels = None
    plotted_image = None
    plt.figure(figsize=(8, 6))
    # plt.imshow(plotted_image)
    # plt.axis('off')
    # if title is not None:
    #     plt.title(title)
    # plt.show()
    return plotted_image

manual_plot_image = None

print_shape('boxes_xyxy', boxes_xyxy)
print('class labels:', class_labels)
print('confidences:', confidences)


### Checks (Exercise 4)

In [ ]:
assert boxes_xyxy.ndim == 2 and boxes_xyxy.shape[1] == 4
assert class_ids.ndim == 1
assert confidences.ndim == 1
assert len(class_ids) == len(confidences) == len(boxes_xyxy)
assert len(class_labels) == len(class_ids)
assert isinstance(result_df, pd.DataFrame)
assert isinstance(manual_plot_image, Image.Image)
print('Exercise 4 passed.')


## 7. Exercise 5: Confidence and YOLO NMS Thresholds

YOLO has two important post-processing controls:
- `conf`: removes low-confidence detections,
- `iou`: controls how aggressively NMS removes overlapping boxes.

Task:
- run the same image with different confidence thresholds,
- run the same image with different YOLO NMS IoU thresholds,
- count detections and compare the rendered outputs.

Useful contracts:
- higher `conf` usually means fewer detections,
- lower `iou` means more aggressive NMS,
- higher `iou` allows more overlapping boxes to remain.


In [ ]:
# Exercise 5

conf_values = [0.1, 0.25, 0.5]
conf_rows = []
conf_results = {}

for conf_value in conf_values:
    conf_result = None
    conf_results[conf_value] = conf_result
    conf_rows.append({
        'setting': f'conf={conf_value}',
        'conf': conf_value,
        'iou': CFG['iou'],
        'num_detections': None,
    })

nms_iou_values = [0.3, 0.7, 0.9]
nms_rows = []
nms_results = {}

for iou_value in nms_iou_values:
    nms_result = None
    nms_results[iou_value] = nms_result
    nms_rows.append({
        'setting': f'iou={iou_value}',
        'conf': CFG['conf'],
        'iou': iou_value,
        'num_detections': None,
    })

threshold_df = pd.DataFrame(conf_rows + nms_rows)
threshold_df


### Checks (Exercise 5)

In [ ]:
expected_settings = []
for conf_value in conf_values:
    expected_settings.append(f'conf={conf_value}')
for iou_value in nms_iou_values:
    expected_settings.append(f'iou={iou_value}')

assert list(threshold_df['setting']) == expected_settings
assert threshold_df['num_detections'].isna().sum() == 0

conf_counts = []
for row_index in range(len(conf_values)):
    count = threshold_df.loc[row_index, 'num_detections']
    conf_counts.append(count)

assert conf_counts[0] >= conf_counts[-1]

for iou_value in nms_iou_values:
    assert iou_value in nms_results

print('Exercise 5 passed.')


## 8. Exercise 6: Failure Analysis Across Images

Detectors are not perfect. A good practitioner inspects both successes and failures.

Task:
- run YOLO on both images,
- render each result,
- list detected class names,
- write a short note about missed objects, false positives, duplicates, or class confusion.


In [ ]:
# Exercise 6

all_results = None
analysis_rows = []

# Fill analysis_rows with one row per image.
# Suggested columns: image_index, detected_classes, num_detections, note

analysis_df = pd.DataFrame(analysis_rows)
analysis_df


### Checks (Exercise 6)

In [ ]:
assert isinstance(analysis_df, pd.DataFrame)
assert len(analysis_df) == len(images)

required_columns = ['image_index', 'detected_classes', 'num_detections', 'note']
for column in required_columns:
    assert column in analysis_df.columns

print('Exercise 6 passed.')


## 9. Wrap-Up Questions
1. What information does an object detector output that a classifier does not?
2. Why is box format important?
3. What does IoU measure?
4. Why does NMS remove some boxes but keep others?
5. What happens when confidence threshold is too low or too high?
6. What failure mode did you observe in the YOLO results?
